In [0]:
import pandas as pd
from pyspark.sql import SparkSession

# reading data from source(Github - Sales) (copy link from file - Raw)
df = pd.read_csv("https://raw.githubusercontent.com/Bhevendra/ML-Datasets/refs/heads/main/retail_data/sales.csv")

# creating sparksession
spark = SparkSession.builder.getOrCreate()


# changing spark dataframe to spark dataframe
df = spark.createDataFrame(df)


In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json

# sample one JSON string
sample_json = df.select("product").filter(col("product").isNotNull()).first()[0]

# infer schema
json_schema = schema_of_json(sample_json)

# flatten
df_flat = df.withColumn("product_json", from_json(col("product"), json_schema)) \
    .select("*", "product_json.*") \
    .drop("product", "product_json")

display(df_flat)

In [0]:
df_flat.write.format("delta").mode("overwrite").saveAsTable("batch_1.data.sales")